# Lab Task 6 — PyTorch CNN Architecture

**Source:** *CNN Implementation*, Laboratory Activity 6.

**Original instruction:** Convert the **provided CNN architecture diagram** into a PyTorch CNN architecture.

> **Missing source needed to verify exact compliance:** The lesson refers to `figures/quick_draw.png`. That image was **not provided with the uploaded files**. This is a **working example based on the CNN layers described in the lecture**, **not a confirmed transcription of the missing instructor diagram**. Replace or revise layer types, kernel sizes, channels, and dimensions after comparing the original image.

The code below implements the known lecture demonstration architecture for MNIST-style grayscale 28×28 inputs: two convolution/max-pooling blocks, flattening, and two fully connected layers.


In [1]:
import torch
from torch import nn

torch.manual_seed(42)

class CNNArchitectureExample(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)   # raw logits for 10 classes

model = CNNArchitectureExample()
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


CNNArchitectureExample(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)
Trainable parameters: 421642


In [2]:
# Verify feature-map and output shapes (batch size 4)
example_images = torch.randn(4, 1, 28, 28)
features = model.features(example_images)
logits = model(example_images)
print('Input:', tuple(example_images.shape))
print('Convolution output:', tuple(features.shape))
print('Class scores:', tuple(logits.shape))
print('Predicted class indexes:', logits.argmax(dim=1).tolist())
assert tuple(features.shape) == (4, 64, 7, 7)
assert tuple(logits.shape) == (4, 10)


Input: (4, 1, 28, 28)
Convolution output: (4, 64, 7, 7)
Class scores: (4, 10)
Predicted class indexes: [3, 3, 8, 8]


In [3]:
# Confirm that one training step can execute (SMOKE TEST, not trained accuracy)
example_labels = torch.tensor([2, 0, 4, 9])
criterion = nn.CrossEntropyLoss()  # raw logits expected, no softmax in forward
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer.zero_grad()
loss = criterion(logits, example_labels)
loss.backward()
optimizer.step()
print(f'One test optimization step completed; dummy-label loss = {loss.item():.4f}')


One test optimization step completed; dummy-label loss = 2.3621


**Architecture trace**

| Stage | Output shape (batch-first) |
|---|---|
| Input | `(B, 1, 28, 28)` |
| Conv2d 1→32, 3×3, padding 1 + ReLU | `(B, 32, 28, 28)` |
| MaxPool2d 2×2 | `(B, 32, 14, 14)` |
| Conv2d 32→64, 3×3, padding 1 + ReLU | `(B, 64, 14, 14)` |
| MaxPool2d 2×2 | `(B, 64, 7, 7)` |
| Flatten | `(B, 3136)` |
| Linear 3136→128 + ReLU | `(B, 128)` |
| Linear 128→10 | `(B, 10)` |

**Before submitting this as Task 6:** obtain the instructor's `quick_draw.png` and confirm the diagram's input shape, convolution filters, pooling, and final output. The demonstrated 28×28 architecture is **not guaranteed to be the same architecture**.
